# Numbers behind our check-in slides
Each cell prints one figure from the deck.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="mcp-acc-055-dbg-p-7e23")
db = "`mcp-ss-data-p-5o6i`.vw_accelerate2605_core_v1"


### Cell 1 - total people, and how many have cancer

In [ ]:
# total people, and how many have a confirmed cancer
sql = f"""
SELECT
  (SELECT COUNT(DISTINCT PATIENT_DK) FROM {db}.DIM_PATIENT) AS total_people,
  (SELECT COUNT(DISTINCT PATIENT_DK) FROM {db}.FACT_CANCER_DATA_REPOSITORY) AS cancer_patients
"""
client.query(sql).to_dataframe()


### Cell 2 - cancer patients treated at Mayo, with data from before treatment

In [ ]:
# patients who got cancer drugs, and who also have lab results from BEFORE treatment
sql = f"""
WITH first_tx AS (
  SELECT t.PATIENT_DK, MIN(DATE(t.TREATMENT_DTM)) AS tx_date
  FROM {db}.FACT_TREATMENT_DETAIL t
  JOIN {db}.DIM_MED_NAME m USING (MED_NAME_DK)
  WHERE UPPER(CAST(m.MED_THERAPEUTIC_CLASS_DESCRIPTION AS STRING)) = "ANTINEOPLASTICS"
    AND t.TREATMENT_DTM IS NOT NULL
  GROUP BY 1
)
SELECT COUNT(DISTINCT f.PATIENT_DK) AS treated_with_before_data
FROM first_tx f
JOIN {db}.FACT_LAB_TEST l
  ON l.PATIENT_DK = f.PATIENT_DK AND DATE(l.LAB_COLLECTION_DTM) < f.tx_date
"""
client.query(sql).to_dataframe()


### Cell 3 - of those, how many have a clear "did it come back" outcome

In [ ]:
# came back = a recurrence was recorded; stayed clear = 2+ years of follow-up after treatment
sql = f"""
WITH first_tx AS (
  SELECT t.PATIENT_DK, MIN(DATE(t.TREATMENT_DTM)) AS tx_date
  FROM {db}.FACT_TREATMENT_DETAIL t
  JOIN {db}.DIM_MED_NAME m USING (MED_NAME_DK)
  WHERE UPPER(CAST(m.MED_THERAPEUTIC_CLASS_DESCRIPTION AS STRING)) = "ANTINEOPLASTICS"
    AND t.TREATMENT_DTM IS NOT NULL
  GROUP BY 1
),
outcome AS (
  SELECT f.PATIENT_DK,
    LOGICAL_OR(c.DATE_RECURRENCE_SUMMARY IS NOT NULL
               AND TRIM(c.DATE_RECURRENCE_SUMMARY) NOT IN ("", "(NONE)")) AS came_back,
    LOGICAL_OR(DATE_DIFF(DATE(c.DATE_LAST_PT_CONTACT_OR_DEATH), f.tx_date, DAY) >= 730) AS followed_2yr
  FROM first_tx f
  JOIN {db}.FACT_CANCER_DATA_REPOSITORY c USING (PATIENT_DK)
  GROUP BY 1
)
SELECT
  COUNTIF(came_back) AS came_back,
  COUNTIF(NOT came_back AND followed_2yr) AS stayed_clear_2yr,
  COUNTIF(came_back OR followed_2yr) AS clear_outcome_total
FROM outcome
"""
client.query(sql).to_dataframe()


### Cell 4 - lab results and vital signs

In [ ]:
# how many lab results we have (billions, going back years)
sql = f"SELECT COUNT(*) AS lab_results FROM {db}.FACT_LAB_TEST"
client.query(sql).to_dataframe()


### Cell 5 - doctors' notes

In [ ]:
# how many clinical notes / documents
sql = f"SELECT COUNT(*) AS clinical_notes FROM {db}.FACT_CLINICAL_DOCUMENTS"
client.query(sql).to_dataframe()


### Cell 6 - medical images

In [ ]:
# how many medical images, and how many patients have them
sql = f"""
SELECT COUNT(*) AS medical_images,
       COUNT(DISTINCT PATIENT_CLINIC_NUMBER) AS patients_with_images
FROM {db}.FACT_RADIOLOGY_DICOM_INVENTORY
"""
client.query(sql).to_dataframe()
